# Facilities Silver: Six-Source Facility Merge

Six-way merge of all facility bronze tables: NHFR (1,359), infirmary (614), hospital list (1,336), primary care (1,180), dialysis clinics (800), and blood service (121) from `public_health.bronze`. Joined on `UPPER(TRIM(facility_name))` with ROW_NUMBER dedup per source. Unmatched records from any source are included with source flags (e.g. `PRIMARY_CARE`, `DIALYSIS`, `BLOOD_SERVICE`). Overlapping fields across all sources are verified with traceable comparison columns.

In [0]:
CREATE SCHEMA IF NOT EXISTS public_health.silver;

CREATE OR REPLACE TEMP VIEW _merge_full AS
WITH nhfr_base AS (
  SELECT
    Health_Facility_Code,
    Health_Facility_Code_Short,
    Facility_Name,
    Old_Health_Facility_Name_1,
    Old_Health_Facility_Name_2,
    Old_Health_Facility_Name_3,
    Facility_Major_Type,
    Health_Facility_Type,
    Ownership_Major_Classification,
    `Ownership_Sub-Classification_for_Government_facilities`,
    `Ownership_Sub-Classification_for_private_facilities`,
    Region_Name,
    Region_PSGC,
    Province_Name,
    Province_PSGC,
    `City/Municipality_Name`,
    `City/Municipality_PSGC`,
    Barangay_Name,
    Barangay_PSGC,
    Zip_Code,
    Service_Capability,
    Bed_Capacity,
    -- Normalized region for cross-source verification
    CASE TRIM(Region_Name)
      WHEN 'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO (BARMM)' THEN 'BARMM'
      WHEN 'CORDILLERA ADMINISTRATIVE REGION (CAR)' THEN 'CAR'
      WHEN 'NATIONAL CAPITAL REGION (NCR)' THEN 'NCR'
      WHEN 'NEGROS ISLAND REGION (NIR)' THEN 'NIR'
      WHEN 'REGION I (ILOCOS REGION)' THEN 'R1'
      WHEN 'REGION II (CAGAYAN VALLEY)' THEN 'R2'
      WHEN 'REGION III (CENTRAL LUZON)' THEN 'R3'
      WHEN 'REGION IV-A (CALABARZON)' THEN 'R4A'
      WHEN 'MIMAROPA REGION' THEN 'R4B'
      WHEN 'REGION V (BICOL REGION)' THEN 'R5'
      WHEN 'REGION VI (WESTERN VISAYAS)' THEN 'R6'
      WHEN 'REGION VII (CENTRAL VISAYAS)' THEN 'R7'
      WHEN 'REGION VIII (EASTERN VISAYAS)' THEN 'R8'
      WHEN 'REGION IX (ZAMBOANGA PENINSULA)' THEN 'R9'
      WHEN 'REGION X (NORTHERN MINDANAO)' THEN 'R10'
      WHEN 'REGION XI (DAVAO REGION)' THEN 'R11'
      WHEN 'REGION XII (SOCCSKSARGEN)' THEN 'R12'
      WHEN 'REGION XIII (CARAGA)' THEN 'R13'
      ELSE NULL
    END AS _n_region_norm,
    UPPER(TRIM(Province_Name))  AS _n_province_norm,
    UPPER(TRIM(`City/Municipality_Name`)) AS _n_city_norm,
    UPPER(TRIM(Ownership_Major_Classification)) AS _n_ownership_norm,
    UPPER(TRIM(`Ownership_Sub-Classification_for_Government_facilities`)) AS _n_ownership_sub_norm
  FROM public_health.bronze.facilities_nhfr_bronze
),
infirmary_base AS (
  SELECT
    nhfr_code,
    facility_name,
    region,
    province,
    city_municipality,
    ownership,
    ownership_class,
    abc,
    add_on_services,
    ambulance_details,
    ambulance_provider,
    CASE TRIM(region)
      WHEN 'BARMM' THEN 'BARMM'
      WHEN 'CORDILLERA' THEN 'CAR'
      WHEN 'NCR' THEN 'NCR'
      WHEN 'NIR' THEN 'NIR'
      WHEN 'REGION 1' THEN 'R1'
      WHEN 'REGION 2' THEN 'R2'
      WHEN 'REGION 3' THEN 'R3'
      WHEN 'REGION 4-A' THEN 'R4A'
      WHEN 'REGION 4-B' THEN 'R4B'
      WHEN 'REGION 5' THEN 'R5'
      WHEN 'REGION 6' THEN 'R6'
      WHEN 'REGION 7' THEN 'R7'
      WHEN 'REGION 8' THEN 'R8'
      WHEN 'REGION 9' THEN 'R9'
      WHEN 'REGION 10' THEN 'R10'
      WHEN 'REGION 11' THEN 'R11'
      WHEN 'REGION 12' THEN 'R12'
      WHEN 'REGION 13' THEN 'R13'
      ELSE NULL
    END AS _i_region_norm,
    UPPER(TRIM(province))         AS _i_province_norm,
    UPPER(TRIM(city_municipality)) AS _i_city_norm,
    UPPER(TRIM(ownership))        AS _i_ownership_norm,
    UPPER(TRIM(ownership_class))  AS _i_ownership_sub_norm
  FROM public_health.bronze.facilities_hfsrb_infirmaries_bronze
)
SELECT
  -- === NHFR COLUMNS (as requested) ===
  n.Health_Facility_Code,
  n.Health_Facility_Code_Short,
  COALESCE(n.Facility_Name, i.facility_name) AS Facility_Name,
  n.Old_Health_Facility_Name_1,
  n.Old_Health_Facility_Name_2,
  n.Old_Health_Facility_Name_3,
  n.Facility_Major_Type,
  n.Health_Facility_Type,
  n.Ownership_Major_Classification,
  n.`Ownership_Sub-Classification_for_Government_facilities`,
  n.`Ownership_Sub-Classification_for_private_facilities`,
  n.Region_Name,
  n.Region_PSGC,
  n.Province_Name,
  n.Province_PSGC,
  n.`City/Municipality_Name`,
  n.`City/Municipality_PSGC`,
  n.Barangay_Name,
  n.Barangay_PSGC,
  n.Zip_Code,
  n.Service_Capability,
  n.Bed_Capacity,

  -- === INFERMARY COLUMNS (as requested) ===
  i.nhfr_code          AS NHFR_Code,
  i.abc                AS ABC,
  i.add_on_services    AS Add_Ons_Services,
  i.ambulance_details  AS Ambulance_Details,
  i.ambulance_provider AS Ambulance_Service_Provider,

  -- ===== TRACEABLE VERIFICATION COLUMNS =====
  -- Merge source flag
  CASE
    WHEN n.Health_Facility_Code IS NOT NULL AND i.facility_name IS NOT NULL THEN 'MATCHED'
    WHEN n.Health_Facility_Code IS NOT NULL THEN 'NHFR_ONLY'
    ELSE 'INFERMARY_ONLY'
  END AS _merge_source,

  -- Region verification (normalized comparison)
  CASE
    WHEN n.Health_Facility_Code IS NULL THEN 'NO_NHFR_MATCH'
    WHEN i.facility_name IS NULL THEN 'NO_INFERMARY_MATCH'
    WHEN n._n_region_norm = i._i_region_norm THEN 'MATCH'
    ELSE 'MISMATCH'
  END AS _verify_region_status,
  n.Region_Name AS _verify_region_nhfr_value,
  i.region      AS _verify_region_infirmary_value,

  -- Province verification
  CASE
    WHEN n.Health_Facility_Code IS NULL THEN 'NO_NHFR_MATCH'
    WHEN i.facility_name IS NULL THEN 'NO_INFERMARY_MATCH'
    WHEN n._n_province_norm = i._i_province_norm THEN 'MATCH'
    ELSE 'MISMATCH'
  END AS _verify_province_status,
  n.Province_Name AS _verify_province_nhfr_value,
  i.province      AS _verify_province_infirmary_value,

  -- City/Municipality verification
  CASE
    WHEN n.Health_Facility_Code IS NULL THEN 'NO_NHFR_MATCH'
    WHEN i.facility_name IS NULL THEN 'NO_INFERMARY_MATCH'
    WHEN n._n_city_norm = i._i_city_norm THEN 'MATCH'
    ELSE 'MISMATCH'
  END AS _verify_city_status,
  n.`City/Municipality_Name` AS _verify_city_nhfr_value,
  i.city_municipality        AS _verify_city_infirmary_value,

  -- Ownership verification
  CASE
    WHEN n.Health_Facility_Code IS NULL THEN 'NO_NHFR_MATCH'
    WHEN i.facility_name IS NULL THEN 'NO_INFERMARY_MATCH'
    WHEN n._n_ownership_norm = i._i_ownership_norm THEN 'MATCH'
    ELSE 'MISMATCH'
  END AS _verify_ownership_status,
  n.Ownership_Major_Classification AS _verify_ownership_nhfr_value,
  i.ownership                      AS _verify_ownership_infirmary_value,

  -- Ownership Sub-Classification verification (government)
  CASE
    WHEN n.Health_Facility_Code IS NULL THEN 'NO_NHFR_MATCH'
    WHEN i.facility_name IS NULL THEN 'NO_INFERMARY_MATCH'
    WHEN i.ownership_class IS NULL AND n.`Ownership_Sub-Classification_for_Government_facilities` IS NULL THEN 'MATCH'
    WHEN n._n_ownership_sub_norm = i._i_ownership_sub_norm THEN 'MATCH'
    ELSE 'MISMATCH'
  END AS _verify_ownership_class_status,
  n.`Ownership_Sub-Classification_for_Government_facilities` AS _verify_ownership_class_nhfr_value,
  i.ownership_class                                          AS _verify_ownership_class_infirmary_value

FROM nhfr_base n
FULL OUTER JOIN infirmary_base i
  ON UPPER(TRIM(n.Facility_Name)) = UPPER(TRIM(i.facility_name));

-- Main merged table (data columns only, as requested)
CREATE OR REPLACE TABLE public_health.silver.facilities_hospital_infirmary_merged AS
SELECT
  Facility_Name,
  Health_Facility_Code,
  Health_Facility_Code_Short,
  Old_Health_Facility_Name_1,
  Old_Health_Facility_Name_2,
  Old_Health_Facility_Name_3,
  Facility_Major_Type,
  Health_Facility_Type,
  Ownership_Major_Classification,
  `Ownership_Sub-Classification_for_Government_facilities`,
  `Ownership_Sub-Classification_for_private_facilities`,
  Region_Name,
  Region_PSGC,
  Province_Name,
  Province_PSGC,
  `City/Municipality_Name`,
  `City/Municipality_PSGC`,
  Barangay_Name,
  Barangay_PSGC,
  Zip_Code,
  Service_Capability,
  Bed_Capacity,
  NHFR_Code,
  ABC,
  Add_Ons_Services,
  Ambulance_Details,
  Ambulance_Service_Provider
FROM _merge_full;

-- Verification table (traceable cross-source comparison)
CREATE OR REPLACE TABLE public_health.silver.verification AS
SELECT
  Facility_Name,
  Health_Facility_Code,
  NHFR_Code AS Infirmary_NHFR_Code,
  _merge_source,
  _verify_region_status,
  _verify_region_nhfr_value,
  _verify_region_infirmary_value,
  _verify_province_status,
  _verify_province_nhfr_value,
  _verify_province_infirmary_value,
  _verify_city_status,
  _verify_city_nhfr_value,
  _verify_city_infirmary_value,
  _verify_ownership_status,
  _verify_ownership_nhfr_value,
  _verify_ownership_infirmary_value,
  _verify_ownership_class_status,
  _verify_ownership_class_nhfr_value,
  _verify_ownership_class_infirmary_value
FROM _merge_full;

-- Clean up temp view
DROP VIEW IF EXISTS _merge_full;

In [0]:
SELECT *
FROM public_health.silver.facilities_hospital_infirmary_merged
WHERE Facility_Name IN (
  SELECT Facility_Name FROM public_health.silver.verification
  WHERE _merge_source LIKE '%+%'
)
ORDER BY Facility_Name
LIMIT 20

In [0]:
SELECT
  _merge_source,
  COUNT(*) AS record_count,
  SUM(CASE WHEN _verify_region_status         NOT IN ('MISMATCH','NA') THEN 1 ELSE 0 END) AS region_match,
  SUM(CASE WHEN _verify_region_status         = 'MISMATCH' THEN 1 ELSE 0 END) AS region_mismatch,
  SUM(CASE WHEN _verify_province_status       NOT IN ('MISMATCH','NA') THEN 1 ELSE 0 END) AS province_match,
  SUM(CASE WHEN _verify_province_status       = 'MISMATCH' THEN 1 ELSE 0 END) AS province_mismatch,
  SUM(CASE WHEN _verify_city_status           NOT IN ('MISMATCH','NA') THEN 1 ELSE 0 END) AS city_match,
  SUM(CASE WHEN _verify_city_status           = 'MISMATCH' THEN 1 ELSE 0 END) AS city_mismatch,
  SUM(CASE WHEN _verify_ownership_status      NOT IN ('MISMATCH','NA') THEN 1 ELSE 0 END) AS ownership_match,
  SUM(CASE WHEN _verify_ownership_status      = 'MISMATCH' THEN 1 ELSE 0 END) AS ownership_mismatch,
  SUM(CASE WHEN _verify_ownership_class_status NOT IN ('MISMATCH','NA') THEN 1 ELSE 0 END) AS own_class_match,
  SUM(CASE WHEN _verify_ownership_class_status = 'MISMATCH' THEN 1 ELSE 0 END) AS own_class_mismatch,
  SUM(CASE WHEN _verify_service_cap_status    NOT IN ('MISMATCH','NA') THEN 1 ELSE 0 END) AS svc_cap_match,
  SUM(CASE WHEN _verify_service_cap_status    = 'MISMATCH' THEN 1 ELSE 0 END) AS svc_cap_mismatch,
  SUM(CASE WHEN _verify_abc_status           NOT IN ('MISMATCH','NA') THEN 1 ELSE 0 END) AS abc_match,
  SUM(CASE WHEN _verify_abc_status           = 'MISMATCH' THEN 1 ELSE 0 END) AS abc_mismatch,
  SUM(CASE WHEN _verify_addons_status        NOT IN ('MISMATCH','NA') THEN 1 ELSE 0 END) AS addons_match,
  SUM(CASE WHEN _verify_addons_status        = 'MISMATCH' THEN 1 ELSE 0 END) AS addons_mismatch,
  SUM(CASE WHEN _verify_ambulance_status     NOT IN ('MISMATCH','NA') THEN 1 ELSE 0 END) AS amb_match,
  SUM(CASE WHEN _verify_ambulance_status     = 'MISMATCH' THEN 1 ELSE 0 END) AS amb_mismatch,
  SUM(CASE WHEN _verify_ambulance_provider_status NOT IN ('MISMATCH','NA') THEN 1 ELSE 0 END) AS amb_prov_match,
  SUM(CASE WHEN _verify_ambulance_provider_status = 'MISMATCH' THEN 1 ELSE 0 END) AS amb_prov_mismatch
FROM public_health.silver.verification
GROUP BY _merge_source
ORDER BY _merge_source

In [0]:
SELECT
  Facility_Name,
  Health_Facility_Code,
  _merge_source,
  _verify_region_status,
  _verify_region_nhfr_value,
  _verify_region_infirmary_value,
  _verify_region_hospital_value,
  _verify_region_primary_care_value,
  _verify_region_dialysis_value,
  _verify_region_blood_service_value,
  _verify_ownership_status,
  _verify_ownership_nhfr_value,
  _verify_ownership_infirmary_value,
  _verify_ownership_hospital_value,
  _verify_ownership_primary_care_value,
  _verify_ownership_dialysis_value,
  _verify_ownership_blood_service_value
FROM public_health.silver.verification
WHERE _merge_source LIKE '%+%'
ORDER BY _merge_source, _verify_region_status, Facility_Name
LIMIT 20

In [0]:
CREATE OR REPLACE TABLE public_health.silver.facilities_infirmary_unmatched AS
SELECT
  i.facility_name   AS Facility_Name,
  i.nhfr_code       AS NHFR_Code,
  i.region,
  i.province,
  i.city_municipality,
  i.address,
  i.ownership,
  i.ownership_class,
  i.abc,
  i.add_on_services,
  i.clinical_lab_class,
  i.head_of_facility,
  i.contact_nos,
  i.email,
  i.ambulance_details,
  i.ambulance_provider,
  i.validity_from,
  i.validity_until,
  i.license_no,
  i.application_status,
  i.remarks
FROM public_health.bronze.facilities_hfsrb_infirmaries_bronze i
WHERE NOT EXISTS (
  SELECT 1 FROM public_health.bronze.facilities_nhfr_bronze n
  WHERE UPPER(TRIM(n.Facility_Name)) = UPPER(TRIM(i.facility_name))
)
AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_hospitals_bronze h WHERE UPPER(TRIM(h.facility_name)) = UPPER(TRIM(i.facility_name)))
AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_primary_care_bronze p WHERE UPPER(TRIM(p.facility_name)) = UPPER(TRIM(i.facility_name)))
AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_dialysis_clinics_bronze d WHERE UPPER(TRIM(d.facility_name)) = UPPER(TRIM(i.facility_name)))
AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_blood_service_bronze b WHERE UPPER(TRIM(b.facility_name)) = UPPER(TRIM(i.facility_name)));

In [0]:
SELECT
  Facility_Name,
  NHFR_Code,
  region,
  province,
  city_municipality,
  ownership,
  ownership_class,
  abc,
  add_on_services,
  ambulance_details,
  ambulance_provider
FROM public_health.silver.facilities_infirmary_unmatched
ORDER BY region, province, Facility_Name
LIMIT 20

In [0]:
CREATE OR REPLACE TABLE public_health.silver.facilities_nhfr_unmatched AS
SELECT
  n.Facility_Name,
  n.Health_Facility_Code,
  n.Health_Facility_Code_Short,
  n.Old_Health_Facility_Name_1,
  n.Old_Health_Facility_Name_2,
  n.Old_Health_Facility_Name_3,
  n.Facility_Major_Type,
  n.Health_Facility_Type,
  n.Ownership_Major_Classification,
  n.`Ownership_Sub-Classification_for_Government_facilities`,
  n.`Ownership_Sub-Classification_for_private_facilities`,
  n.Region_Name,
  n.Region_PSGC,
  n.Province_Name,
  n.Province_PSGC,
  n.`City/Municipality_Name`,
  n.`City/Municipality_PSGC`,
  n.Barangay_Name,
  n.Barangay_PSGC,
  n.Zip_Code,
  n.Service_Capability,
  n.Bed_Capacity
FROM public_health.bronze.facilities_nhfr_bronze n
WHERE NOT EXISTS (
  SELECT 1 FROM public_health.bronze.facilities_hfsrb_infirmaries_bronze i
  WHERE UPPER(TRIM(i.facility_name)) = UPPER(TRIM(n.Facility_Name))
)
AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_hospitals_bronze h WHERE UPPER(TRIM(h.facility_name)) = UPPER(TRIM(n.Facility_Name)))
AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_primary_care_bronze p WHERE UPPER(TRIM(p.facility_name)) = UPPER(TRIM(n.Facility_Name)))
AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_dialysis_clinics_bronze d WHERE UPPER(TRIM(d.facility_name)) = UPPER(TRIM(n.Facility_Name)))
AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_blood_service_bronze b WHERE UPPER(TRIM(b.facility_name)) = UPPER(TRIM(n.Facility_Name)));

In [0]:
SELECT
  Facility_Name,
  Health_Facility_Code,
  Health_Facility_Code_Short,
  Region_Name,
  Province_Name,
  `City/Municipality_Name`,
  Ownership_Major_Classification,
  Health_Facility_Type,
  Service_Capability,
  Bed_Capacity
FROM public_health.silver.facilities_nhfr_unmatched
ORDER BY Region_Name, Province_Name, Facility_Name
LIMIT 20

In [0]:
CREATE OR REPLACE TEMP VIEW _merge_six AS
WITH all_names AS (
  SELECT DISTINCT UPPER(TRIM(Facility_Name)) AS join_key FROM public_health.bronze.facilities_nhfr_bronze WHERE Facility_Name IS NOT NULL
  UNION
  SELECT DISTINCT UPPER(TRIM(facility_name)) FROM public_health.bronze.facilities_hfsrb_infirmaries_bronze WHERE facility_name IS NOT NULL
  UNION
  SELECT DISTINCT UPPER(TRIM(facility_name)) FROM public_health.bronze.facilities_hfsrb_hospitals_bronze WHERE facility_name IS NOT NULL
  UNION
  SELECT DISTINCT UPPER(TRIM(facility_name)) FROM public_health.bronze.facilities_hfsrb_primary_care_bronze WHERE facility_name IS NOT NULL
  UNION
  SELECT DISTINCT UPPER(TRIM(facility_name)) FROM public_health.bronze.facilities_hfsrb_dialysis_clinics_bronze WHERE facility_name IS NOT NULL
  UNION
  SELECT DISTINCT UPPER(TRIM(facility_name)) FROM public_health.bronze.facilities_hfsrb_blood_service_bronze WHERE facility_name IS NOT NULL
),
nhfr_base AS (
  SELECT * FROM (
    SELECT
      Health_Facility_Code, Health_Facility_Code_Short, Facility_Name,
      Old_Health_Facility_Name_1, Old_Health_Facility_Name_2, Old_Health_Facility_Name_3,
      Facility_Major_Type, Health_Facility_Type, Ownership_Major_Classification,
      `Ownership_Sub-Classification_for_Government_facilities`,
      `Ownership_Sub-Classification_for_private_facilities`,
      Region_Name, Region_PSGC, Province_Name, Province_PSGC,
      `City/Municipality_Name`, `City/Municipality_PSGC`,
      Barangay_Name, Barangay_PSGC, Zip_Code, Service_Capability, Bed_Capacity,
      CASE TRIM(Region_Name)
        WHEN 'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO (BARMM)' THEN 'BARMM'
        WHEN 'CORDILLERA ADMINISTRATIVE REGION (CAR)' THEN 'CAR'
        WHEN 'NATIONAL CAPITAL REGION (NCR)' THEN 'NCR'
        WHEN 'NEGROS ISLAND REGION (NIR)' THEN 'NIR'
        WHEN 'REGION I (ILOCOS REGION)' THEN 'R1'
        WHEN 'REGION II (CAGAYAN VALLEY)' THEN 'R2'
        WHEN 'REGION III (CENTRAL LUZON)' THEN 'R3'
        WHEN 'REGION IV-A (CALABARZON)' THEN 'R4A'
        WHEN 'MIMAROPA REGION' THEN 'R4B'
        WHEN 'REGION V (BICOL REGION)' THEN 'R5'
        WHEN 'REGION VI (WESTERN VISAYAS)' THEN 'R6'
        WHEN 'REGION VII (CENTRAL VISAYAS)' THEN 'R7'
        WHEN 'REGION VIII (EASTERN VISAYAS)' THEN 'R8'
        WHEN 'REGION IX (ZAMBOANGA PENINSULA)' THEN 'R9'
        WHEN 'REGION X (NORTHERN MINDANAO)' THEN 'R10'
        WHEN 'REGION XI (DAVAO REGION)' THEN 'R11'
        WHEN 'REGION XII (SOCCSKSARGEN)' THEN 'R12'
        WHEN 'REGION XIII (CARAGA)' THEN 'R13'
        ELSE NULL
      END AS _n_region_norm,
      UPPER(TRIM(Province_Name)) AS _n_province_norm,
      UPPER(TRIM(`City/Municipality_Name`)) AS _n_city_norm,
      UPPER(TRIM(Ownership_Major_Classification)) AS _n_ownership_norm,
      UPPER(TRIM(`Ownership_Sub-Classification_for_Government_facilities`)) AS _n_ownership_sub_norm,
      UPPER(TRIM(Service_Capability)) AS _n_service_cap_norm,
      ROW_NUMBER() OVER (PARTITION BY UPPER(TRIM(Facility_Name)) ORDER BY Health_Facility_Code) AS _rn
    FROM public_health.bronze.facilities_nhfr_bronze
  ) WHERE _rn = 1
),
infirmary_base AS (
  SELECT * FROM (
    SELECT
      nhfr_code, facility_name, region, province, city_municipality,
      ownership, ownership_class, abc, add_on_services, ambulance_details, ambulance_provider,
      CASE UPPER(TRIM(region))
        WHEN 'BARMM' THEN 'BARMM' WHEN 'CORDILLERA' THEN 'CAR' WHEN 'NCR' THEN 'NCR'
        WHEN 'NIR' THEN 'NIR' WHEN 'REGION 1' THEN 'R1' WHEN 'REGION 2' THEN 'R2'
        WHEN 'REGION 3' THEN 'R3' WHEN 'REGION 4-A' THEN 'R4A' WHEN 'REGION 4-B' THEN 'R4B'
        WHEN 'REGION 5' THEN 'R5' WHEN 'REGION 6' THEN 'R6' WHEN 'REGION 7' THEN 'R7'
        WHEN 'REGION 8' THEN 'R8' WHEN 'REGION 9' THEN 'R9' WHEN 'REGION 10' THEN 'R10'
        WHEN 'REGION 11' THEN 'R11' WHEN 'REGION 12' THEN 'R12' WHEN 'REGION 13' THEN 'R13'
        ELSE NULL
      END AS _i_region_norm,
      UPPER(TRIM(province)) AS _i_province_norm,
      UPPER(TRIM(city_municipality)) AS _i_city_norm,
      UPPER(TRIM(ownership)) AS _i_ownership_norm,
      UPPER(TRIM(ownership_class)) AS _i_ownership_sub_norm,
      ROW_NUMBER() OVER (PARTITION BY UPPER(TRIM(facility_name)) ORDER BY nhfr_code) AS _rn
    FROM public_health.bronze.facilities_hfsrb_infirmaries_bronze
  ) WHERE _rn = 1
),
hospital_base AS (
  SELECT * FROM (
    SELECT
      nhfr_code, facility_name, region, province, city_municipality,
      ownership, ownership_class, hospital_class, service_capability, abc,
      add_on_services, blood_service_facility, rhiv_confirmatory_lab,
      ambulance_details, ambulance_type1_count, ambulance_type2_count, ambulance_total,
      ambulance_provider, dialysis_stations, dialysis_provider,
      ctf_classification, clinical_lab_class,
      CASE UPPER(TRIM(region))
        WHEN 'BARMM' THEN 'BARMM' WHEN 'CORDILLERA' THEN 'CAR' WHEN 'NCR' THEN 'NCR'
        WHEN 'NIR' THEN 'NIR' WHEN 'REGION 1' THEN 'R1' WHEN 'REGION 2' THEN 'R2'
        WHEN 'REGION 3' THEN 'R3' WHEN 'REGION 4-A' THEN 'R4A' WHEN 'REGION 4-B' THEN 'R4B'
        WHEN 'REGION 5' THEN 'R5' WHEN 'REGION 6' THEN 'R6' WHEN 'REGION 7' THEN 'R7'
        WHEN 'REGION 8' THEN 'R8' WHEN 'REGION 9' THEN 'R9' WHEN 'REGION 10' THEN 'R10'
        WHEN 'REGION 11' THEN 'R11' WHEN 'REGION 12' THEN 'R12' WHEN 'REGION 13' THEN 'R13'
        ELSE NULL
      END AS _h_region_norm,
      UPPER(TRIM(province)) AS _h_province_norm,
      UPPER(TRIM(city_municipality)) AS _h_city_norm,
      UPPER(TRIM(ownership)) AS _h_ownership_norm,
      UPPER(TRIM(ownership_class)) AS _h_ownership_sub_norm,
      UPPER(TRIM(service_capability)) AS _h_service_cap_norm,
      ROW_NUMBER() OVER (PARTITION BY UPPER(TRIM(facility_name)) ORDER BY nhfr_code) AS _rn
    FROM public_health.bronze.facilities_hfsrb_hospitals_bronze
  ) WHERE _rn = 1
),
primary_care_base AS (
  SELECT * FROM (
    SELECT
      nhfr_code, facility_name, region, province, city_municipality,
      ownership, head_of_facility,
      ambulance_arrangement, ambulance_provider, ambulance_class,
      birthing_home_arrangement, birthing_home_abc, birthing_home_provider,
      clinical_lab_arrangement, clinical_lab_capability, clinical_lab_provider,
      dental_arrangement, dental_provider,
      pharmacy_arrangement, pharmacy_provider,
      xray_arrangement, xray_provider, ancillary_services,
      CASE UPPER(TRIM(region))
        WHEN 'BARMM' THEN 'BARMM' WHEN 'CORDILLERA' THEN 'CAR' WHEN 'NCR' THEN 'NCR'
        WHEN 'NIR' THEN 'NIR' WHEN 'REGION 1' THEN 'R1' WHEN 'REGION 2' THEN 'R2'
        WHEN 'REGION 3' THEN 'R3' WHEN 'REGION 4-A' THEN 'R4A' WHEN 'REGION 4-B' THEN 'R4B'
        WHEN 'REGION 5' THEN 'R5' WHEN 'REGION 6' THEN 'R6' WHEN 'REGION 7' THEN 'R7'
        WHEN 'REGION 8' THEN 'R8' WHEN 'REGION 9' THEN 'R9' WHEN 'REGION 10' THEN 'R10'
        WHEN 'REGION 11' THEN 'R11' WHEN 'REGION 12' THEN 'R12' WHEN 'REGION 13' THEN 'R13'
        ELSE NULL
      END AS _pc_region_norm,
      UPPER(TRIM(province)) AS _pc_province_norm,
      UPPER(TRIM(city_municipality)) AS _pc_city_norm,
      UPPER(TRIM(ownership)) AS _pc_ownership_norm,
      ROW_NUMBER() OVER (PARTITION BY UPPER(TRIM(facility_name)) ORDER BY _source_row_num) AS _rn
    FROM public_health.bronze.facilities_hfsrb_primary_care_bronze
  ) WHERE _rn = 1
),
dialysis_base AS (
  SELECT * FROM (
    SELECT
      nhfr_code, facility_name, region, province, city_municipality,
      ownership, institutional_character, stations, head_of_facility,
      CASE UPPER(TRIM(region))
        WHEN 'BARMM' THEN 'BARMM' WHEN 'CORDILLERA' THEN 'CAR' WHEN 'NCR' THEN 'NCR'
        WHEN 'NIR' THEN 'NIR' WHEN 'REGION 1' THEN 'R1' WHEN 'REGION 2' THEN 'R2'
        WHEN 'REGION 3' THEN 'R3' WHEN 'REGION 4-A' THEN 'R4A' WHEN 'REGION 4-B' THEN 'R4B'
        WHEN 'REGION 5' THEN 'R5' WHEN 'REGION 6' THEN 'R6' WHEN 'REGION 7' THEN 'R7'
        WHEN 'REGION 8' THEN 'R8' WHEN 'REGION 9' THEN 'R9' WHEN 'REGION 10' THEN 'R10'
        WHEN 'REGION 11' THEN 'R11' WHEN 'REGION 12' THEN 'R12' WHEN 'REGION 13' THEN 'R13'
        ELSE NULL
      END AS _dc_region_norm,
      UPPER(TRIM(province)) AS _dc_province_norm,
      UPPER(TRIM(city_municipality)) AS _dc_city_norm,
      UPPER(TRIM(ownership)) AS _dc_ownership_norm,
      ROW_NUMBER() OVER (PARTITION BY UPPER(TRIM(facility_name)) ORDER BY _source_row_num) AS _rn
    FROM public_health.bronze.facilities_hfsrb_dialysis_clinics_bronze
  ) WHERE _rn = 1
),
blood_service_base AS (
  SELECT
    nhfr_code, facility_name, region, province, city_municipality,
    ownership, service_capability, institutional_character, head_of_laboratory,
    CASE UPPER(TRIM(region))
      WHEN 'BARMM' THEN 'BARMM' WHEN 'CORDILLERA' THEN 'CAR' WHEN 'NCR' THEN 'NCR'
      WHEN 'NIR' THEN 'NIR' WHEN 'REGION 1' THEN 'R1' WHEN 'REGION 2' THEN 'R2'
      WHEN 'REGION 3' THEN 'R3' WHEN 'REGION 4-A' THEN 'R4A' WHEN 'REGION 4-B' THEN 'R4B'
      WHEN 'REGION 5' THEN 'R5' WHEN 'REGION 6' THEN 'R6' WHEN 'REGION 7' THEN 'R7'
      WHEN 'REGION 8' THEN 'R8' WHEN 'REGION 9' THEN 'R9' WHEN 'REGION 10' THEN 'R10'
      WHEN 'REGION 11' THEN 'R11' WHEN 'REGION 12' THEN 'R12' WHEN 'REGION 13' THEN 'R13'
      ELSE NULL
    END AS _bs_region_norm,
    UPPER(TRIM(province)) AS _bs_province_norm,
    UPPER(TRIM(city_municipality)) AS _bs_city_norm,
    UPPER(TRIM(ownership)) AS _bs_ownership_norm,
    UPPER(TRIM(service_capability)) AS _bs_service_cap_norm
  FROM public_health.bronze.facilities_hfsrb_blood_service_bronze
)
SELECT
  COALESCE(n.Facility_Name, i.facility_name, h.facility_name, pc.facility_name, dc.facility_name, bs.facility_name) AS Facility_Name,
  n.Health_Facility_Code, n.Health_Facility_Code_Short,
  n.Old_Health_Facility_Name_1, n.Old_Health_Facility_Name_2, n.Old_Health_Facility_Name_3,
  n.Facility_Major_Type, n.Health_Facility_Type,
  n.Ownership_Major_Classification,
  n.`Ownership_Sub-Classification_for_Government_facilities`,
  n.`Ownership_Sub-Classification_for_private_facilities`,
  n.Region_Name, n.Region_PSGC, n.Province_Name, n.Province_PSGC,
  n.`City/Municipality_Name`, n.`City/Municipality_PSGC`,
  n.Barangay_Name, n.Barangay_PSGC, n.Zip_Code, n.Service_Capability, n.Bed_Capacity,
  i.nhfr_code AS Infirmary_NHFR_Code, i.abc AS Infirmary_ABC,
  i.add_on_services AS Infirmary_Add_Ons_Services,
  i.ambulance_details AS Infirmary_Ambulance_Details,
  i.ambulance_provider AS Infirmary_Ambulance_Service_Provider,
  h.nhfr_code AS Hospital_NHFR_Code, h.hospital_class AS Hospital_Class,
  h.service_capability AS Hospital_Service_Capability, h.abc AS Hospital_ABC,
  h.add_on_services AS Hospital_Add_Ons_Services,
  h.blood_service_facility AS Hospital_Blood_Service_Facility,
  h.rhiv_confirmatory_lab AS Hospital_rHIV_Confirmatory_Lab,
  h.ambulance_details AS Hospital_Ambulance_Details,
  h.ambulance_type1_count AS Hospital_Ambulance_Type1_Count,
  h.ambulance_type2_count AS Hospital_Ambulance_Type2_Count,
  h.ambulance_total AS Hospital_Ambulance_Total,
  h.ambulance_provider AS Hospital_Ambulance_Provider,
  h.dialysis_stations AS Hospital_Dialysis_Stations,
  h.dialysis_provider AS Hospital_Dialysis_Provider,
  h.ctf_classification AS Hospital_CTF_Classification,
  h.clinical_lab_class AS Hospital_Clinical_Lab_Class,
  pc.nhfr_code AS PrimaryCare_NHFR_Code,
  pc.ownership AS PrimaryCare_Ownership,
  pc.head_of_facility AS PrimaryCare_Head_Of_Facility,
  pc.ambulance_arrangement AS PrimaryCare_Ambulance_Arrangement,
  pc.ambulance_provider AS PrimaryCare_Ambulance_Provider,
  pc.birthing_home_abc AS PrimaryCare_Birthing_Home_ABC,
  pc.clinical_lab_capability AS PrimaryCare_Clinical_Lab_Capability,
  pc.pharmacy_arrangement AS PrimaryCare_Pharmacy_Arrangement,
  pc.xray_arrangement AS PrimaryCare_Xray_Arrangement,
  pc.ancillary_services AS PrimaryCare_Ancillary_Services,
  dc.nhfr_code AS Dialysis_NHFR_Code,
  dc.ownership AS Dialysis_Ownership,
  dc.institutional_character AS Dialysis_Institutional_Character,
  dc.stations AS Dialysis_Stations,
  dc.head_of_facility AS Dialysis_Head_Of_Facility,
  bs.nhfr_code AS BloodService_NHFR_Code,
  bs.ownership AS BloodService_Ownership,
  bs.service_capability AS BloodService_Service_Capability,
  bs.institutional_character AS BloodService_Institutional_Character,
  bs.head_of_laboratory AS BloodService_Head_Of_Laboratory,
  TRIM(BOTH '+' FROM CONCAT_WS('+',
    CASE WHEN n.Health_Facility_Code IS NOT NULL THEN 'NHFR' END,
    CASE WHEN i.facility_name IS NOT NULL THEN 'INFERMARY' END,
    CASE WHEN h.facility_name IS NOT NULL THEN 'HOSPITAL' END,
    CASE WHEN pc.facility_name IS NOT NULL THEN 'PRIMARY_CARE' END,
    CASE WHEN dc.facility_name IS NOT NULL THEN 'DIALYSIS' END,
    CASE WHEN bs.facility_name IS NOT NULL THEN 'BLOOD_SERVICE' END
  )) AS _merge_source,
  CASE
    WHEN SIZE(ARRAY_COMPACT(ARRAY(n._n_region_norm, i._i_region_norm, h._h_region_norm, pc._pc_region_norm, dc._dc_region_norm, bs._bs_region_norm))) = 0 THEN 'NA'
    WHEN SIZE(ARRAY_COMPACT(ARRAY(n._n_region_norm, i._i_region_norm, h._h_region_norm, pc._pc_region_norm, dc._dc_region_norm, bs._bs_region_norm))) = 1 THEN 'SINGLE_SOURCE'
    WHEN SIZE(ARRAY_DISTINCT(ARRAY_COMPACT(ARRAY(n._n_region_norm, i._i_region_norm, h._h_region_norm, pc._pc_region_norm, dc._dc_region_norm, bs._bs_region_norm)))) = 1 THEN 'ALL_MATCH'
    ELSE 'MISMATCH'
  END AS _verify_region_status,
  n.Region_Name AS _verify_region_nhfr_value,
  i.region AS _verify_region_infirmary_value,
  h.region AS _verify_region_hospital_value,
  pc.region AS _verify_region_primary_care_value,
  dc.region AS _verify_region_dialysis_value,
  bs.region AS _verify_region_blood_service_value,
  CASE
    WHEN SIZE(ARRAY_COMPACT(ARRAY(n._n_province_norm, i._i_province_norm, h._h_province_norm, pc._pc_province_norm, dc._dc_province_norm, bs._bs_province_norm))) = 0 THEN 'NA'
    WHEN SIZE(ARRAY_COMPACT(ARRAY(n._n_province_norm, i._i_province_norm, h._h_province_norm, pc._pc_province_norm, dc._dc_province_norm, bs._bs_province_norm))) = 1 THEN 'SINGLE_SOURCE'
    WHEN SIZE(ARRAY_DISTINCT(ARRAY_COMPACT(ARRAY(n._n_province_norm, i._i_province_norm, h._h_province_norm, pc._pc_province_norm, dc._dc_province_norm, bs._bs_province_norm)))) = 1 THEN 'ALL_MATCH'
    ELSE 'MISMATCH'
  END AS _verify_province_status,
  n.Province_Name AS _verify_province_nhfr_value,
  i.province AS _verify_province_infirmary_value,
  h.province AS _verify_province_hospital_value,
  pc.province AS _verify_province_primary_care_value,
  dc.province AS _verify_province_dialysis_value,
  bs.province AS _verify_province_blood_service_value,
  CASE
    WHEN SIZE(ARRAY_COMPACT(ARRAY(n._n_city_norm, i._i_city_norm, h._h_city_norm, pc._pc_city_norm, dc._dc_city_norm, bs._bs_city_norm))) = 0 THEN 'NA'
    WHEN SIZE(ARRAY_COMPACT(ARRAY(n._n_city_norm, i._i_city_norm, h._h_city_norm, pc._pc_city_norm, dc._dc_city_norm, bs._bs_city_norm))) = 1 THEN 'SINGLE_SOURCE'
    WHEN SIZE(ARRAY_DISTINCT(ARRAY_COMPACT(ARRAY(n._n_city_norm, i._i_city_norm, h._h_city_norm, pc._pc_city_norm, dc._dc_city_norm, bs._bs_city_norm)))) = 1 THEN 'ALL_MATCH'
    ELSE 'MISMATCH'
  END AS _verify_city_status,
  n.`City/Municipality_Name` AS _verify_city_nhfr_value,
  i.city_municipality AS _verify_city_infirmary_value,
  h.city_municipality AS _verify_city_hospital_value,
  pc.city_municipality AS _verify_city_primary_care_value,
  dc.city_municipality AS _verify_city_dialysis_value,
  bs.city_municipality AS _verify_city_blood_service_value,
  CASE
    WHEN SIZE(ARRAY_COMPACT(ARRAY(n._n_ownership_norm, i._i_ownership_norm, h._h_ownership_norm, pc._pc_ownership_norm, dc._dc_ownership_norm, bs._bs_ownership_norm))) = 0 THEN 'NA'
    WHEN SIZE(ARRAY_COMPACT(ARRAY(n._n_ownership_norm, i._i_ownership_norm, h._h_ownership_norm, pc._pc_ownership_norm, dc._dc_ownership_norm, bs._bs_ownership_norm))) = 1 THEN 'SINGLE_SOURCE'
    WHEN SIZE(ARRAY_DISTINCT(ARRAY_COMPACT(ARRAY(n._n_ownership_norm, i._i_ownership_norm, h._h_ownership_norm, pc._pc_ownership_norm, dc._dc_ownership_norm, bs._bs_ownership_norm)))) = 1 THEN 'ALL_MATCH'
    ELSE 'MISMATCH'
  END AS _verify_ownership_status,
  n.Ownership_Major_Classification AS _verify_ownership_nhfr_value,
  i.ownership AS _verify_ownership_infirmary_value,
  h.ownership AS _verify_ownership_hospital_value,
  pc.ownership AS _verify_ownership_primary_care_value,
  dc.ownership AS _verify_ownership_dialysis_value,
  bs.ownership AS _verify_ownership_blood_service_value,
  CASE
    WHEN n._n_ownership_sub_norm IS NOT NULL AND i._i_ownership_sub_norm IS NOT NULL AND h._h_ownership_sub_norm IS NOT NULL THEN
      CASE WHEN n._n_ownership_sub_norm = i._i_ownership_sub_norm AND n._n_ownership_sub_norm = h._h_ownership_sub_norm THEN 'ALL_MATCH' ELSE 'MISMATCH' END
    WHEN n._n_ownership_sub_norm IS NOT NULL AND i._i_ownership_sub_norm IS NOT NULL THEN
      CASE WHEN n._n_ownership_sub_norm = i._i_ownership_sub_norm THEN 'MATCH' ELSE 'MISMATCH' END
    WHEN n._n_ownership_sub_norm IS NOT NULL AND h._h_ownership_sub_norm IS NOT NULL THEN
      CASE WHEN n._n_ownership_sub_norm = h._h_ownership_sub_norm THEN 'MATCH' ELSE 'MISMATCH' END
    WHEN i._i_ownership_sub_norm IS NOT NULL AND h._h_ownership_sub_norm IS NOT NULL THEN
      CASE WHEN i._i_ownership_sub_norm = h._h_ownership_sub_norm THEN 'MATCH' ELSE 'MISMATCH' END
    ELSE 'SINGLE_SOURCE'
  END AS _verify_ownership_class_status,
  n.`Ownership_Sub-Classification_for_Government_facilities` AS _verify_ownership_class_nhfr_value,
  i.ownership_class AS _verify_ownership_class_infirmary_value,
  h.ownership_class AS _verify_ownership_class_hospital_value,
  CASE
    WHEN SIZE(ARRAY_COMPACT(ARRAY(n._n_service_cap_norm, h._h_service_cap_norm, bs._bs_service_cap_norm))) = 0 THEN 'NA'
    WHEN SIZE(ARRAY_COMPACT(ARRAY(n._n_service_cap_norm, h._h_service_cap_norm, bs._bs_service_cap_norm))) = 1 THEN 'SINGLE_SOURCE'
    WHEN SIZE(ARRAY_DISTINCT(ARRAY_COMPACT(ARRAY(n._n_service_cap_norm, h._h_service_cap_norm, bs._bs_service_cap_norm)))) = 1 THEN 'ALL_MATCH'
    ELSE 'MISMATCH'
  END AS _verify_service_cap_status,
  n.Service_Capability AS _verify_service_cap_nhfr_value,
  h.service_capability AS _verify_service_cap_hospital_value,
  bs.service_capability AS _verify_service_cap_blood_service_value,
  CASE
    WHEN i.abc IS NOT NULL AND h.abc IS NOT NULL THEN
      CASE WHEN UPPER(TRIM(i.abc)) = UPPER(TRIM(h.abc)) THEN 'MATCH' ELSE 'MISMATCH' END
    ELSE 'NA'
  END AS _verify_abc_status,
  i.abc AS _verify_abc_infirmary_value,
  h.abc AS _verify_abc_hospital_value,
  CASE
    WHEN i.add_on_services IS NOT NULL AND h.add_on_services IS NOT NULL THEN
      CASE WHEN UPPER(TRIM(i.add_on_services)) = UPPER(TRIM(h.add_on_services)) THEN 'MATCH' ELSE 'MISMATCH' END
    ELSE 'NA'
  END AS _verify_addons_status,
  i.add_on_services AS _verify_addons_infirmary_value,
  h.add_on_services AS _verify_addons_hospital_value,
  CASE
    WHEN SIZE(ARRAY_COMPACT(ARRAY(UPPER(TRIM(i.ambulance_details)), UPPER(TRIM(h.ambulance_details)), UPPER(TRIM(pc.ambulance_arrangement))))) = 0 THEN 'NA'
    WHEN SIZE(ARRAY_COMPACT(ARRAY(UPPER(TRIM(i.ambulance_details)), UPPER(TRIM(h.ambulance_details)), UPPER(TRIM(pc.ambulance_arrangement))))) = 1 THEN 'SINGLE_SOURCE'
    WHEN SIZE(ARRAY_DISTINCT(ARRAY_COMPACT(ARRAY(UPPER(TRIM(i.ambulance_details)), UPPER(TRIM(h.ambulance_details)), UPPER(TRIM(pc.ambulance_arrangement)))))) = 1 THEN 'ALL_MATCH'
    ELSE 'MISMATCH'
  END AS _verify_ambulance_status,
  i.ambulance_details AS _verify_ambulance_infirmary_value,
  h.ambulance_details AS _verify_ambulance_hospital_value,
  pc.ambulance_arrangement AS _verify_ambulance_primary_care_value,
  CASE
    WHEN SIZE(ARRAY_COMPACT(ARRAY(UPPER(TRIM(i.ambulance_provider)), UPPER(TRIM(h.ambulance_provider)), UPPER(TRIM(pc.ambulance_provider))))) = 0 THEN 'NA'
    WHEN SIZE(ARRAY_COMPACT(ARRAY(UPPER(TRIM(i.ambulance_provider)), UPPER(TRIM(h.ambulance_provider)), UPPER(TRIM(pc.ambulance_provider))))) = 1 THEN 'SINGLE_SOURCE'
    WHEN SIZE(ARRAY_DISTINCT(ARRAY_COMPACT(ARRAY(UPPER(TRIM(i.ambulance_provider)), UPPER(TRIM(h.ambulance_provider)), UPPER(TRIM(pc.ambulance_provider)))))) = 1 THEN 'ALL_MATCH'
    ELSE 'MISMATCH'
  END AS _verify_ambulance_provider_status,
  i.ambulance_provider AS _verify_ambulance_provider_infirmary_value,
  h.ambulance_provider AS _verify_ambulance_provider_hospital_value,
  pc.ambulance_provider AS _verify_ambulance_provider_primary_care_value
FROM all_names a
LEFT JOIN nhfr_base n ON a.join_key = UPPER(TRIM(n.Facility_Name))
LEFT JOIN infirmary_base i ON a.join_key = UPPER(TRIM(i.facility_name))
LEFT JOIN hospital_base h ON a.join_key = UPPER(TRIM(h.facility_name))
LEFT JOIN primary_care_base pc ON a.join_key = UPPER(TRIM(pc.facility_name))
LEFT JOIN dialysis_base dc ON a.join_key = UPPER(TRIM(dc.facility_name))
LEFT JOIN blood_service_base bs ON a.join_key = UPPER(TRIM(bs.facility_name));

CREATE OR REPLACE TABLE public_health.silver.facilities_hospital_infirmary_merged AS
SELECT
  Facility_Name,
  Health_Facility_Code, Health_Facility_Code_Short,
  Old_Health_Facility_Name_1, Old_Health_Facility_Name_2, Old_Health_Facility_Name_3,
  Facility_Major_Type, Health_Facility_Type, Ownership_Major_Classification,
  `Ownership_Sub-Classification_for_Government_facilities`,
  `Ownership_Sub-Classification_for_private_facilities`,
  Region_Name, Region_PSGC, Province_Name, Province_PSGC,
  `City/Municipality_Name`, `City/Municipality_PSGC`,
  Barangay_Name, Barangay_PSGC, Zip_Code, Service_Capability, Bed_Capacity,
  Infirmary_NHFR_Code, Infirmary_ABC, Infirmary_Add_Ons_Services,
  Infirmary_Ambulance_Details, Infirmary_Ambulance_Service_Provider,
  Hospital_NHFR_Code, Hospital_Class, Hospital_Service_Capability, Hospital_ABC,
  Hospital_Add_Ons_Services, Hospital_Blood_Service_Facility,
  Hospital_rHIV_Confirmatory_Lab, Hospital_Ambulance_Details,
  Hospital_Ambulance_Type1_Count, Hospital_Ambulance_Type2_Count,
  Hospital_Ambulance_Total, Hospital_Ambulance_Provider,
  Hospital_Dialysis_Stations, Hospital_Dialysis_Provider,
  Hospital_CTF_Classification, Hospital_Clinical_Lab_Class,
  PrimaryCare_NHFR_Code, PrimaryCare_Ownership, PrimaryCare_Head_Of_Facility,
  PrimaryCare_Ambulance_Arrangement, PrimaryCare_Ambulance_Provider,
  PrimaryCare_Birthing_Home_ABC, PrimaryCare_Clinical_Lab_Capability,
  PrimaryCare_Pharmacy_Arrangement, PrimaryCare_Xray_Arrangement,
  PrimaryCare_Ancillary_Services,
  Dialysis_NHFR_Code, Dialysis_Ownership, Dialysis_Institutional_Character,
  Dialysis_Stations, Dialysis_Head_Of_Facility,
  BloodService_NHFR_Code, BloodService_Ownership, BloodService_Service_Capability,
  BloodService_Institutional_Character, BloodService_Head_Of_Laboratory
FROM _merge_six;

CREATE OR REPLACE TABLE public_health.silver.verification AS
SELECT
  Facility_Name, Health_Facility_Code, _merge_source,
  _verify_region_status, _verify_region_nhfr_value, _verify_region_infirmary_value, _verify_region_hospital_value, _verify_region_primary_care_value, _verify_region_dialysis_value, _verify_region_blood_service_value,
  _verify_province_status, _verify_province_nhfr_value, _verify_province_infirmary_value, _verify_province_hospital_value, _verify_province_primary_care_value, _verify_province_dialysis_value, _verify_province_blood_service_value,
  _verify_city_status, _verify_city_nhfr_value, _verify_city_infirmary_value, _verify_city_hospital_value, _verify_city_primary_care_value, _verify_city_dialysis_value, _verify_city_blood_service_value,
  _verify_ownership_status, _verify_ownership_nhfr_value, _verify_ownership_infirmary_value, _verify_ownership_hospital_value, _verify_ownership_primary_care_value, _verify_ownership_dialysis_value, _verify_ownership_blood_service_value,
  _verify_ownership_class_status, _verify_ownership_class_nhfr_value, _verify_ownership_class_infirmary_value, _verify_ownership_class_hospital_value,
  _verify_service_cap_status, _verify_service_cap_nhfr_value, _verify_service_cap_hospital_value, _verify_service_cap_blood_service_value,
  _verify_abc_status, _verify_abc_infirmary_value, _verify_abc_hospital_value,
  _verify_addons_status, _verify_addons_infirmary_value, _verify_addons_hospital_value,
  _verify_ambulance_status, _verify_ambulance_infirmary_value, _verify_ambulance_hospital_value, _verify_ambulance_primary_care_value,
  _verify_ambulance_provider_status, _verify_ambulance_provider_infirmary_value, _verify_ambulance_provider_hospital_value, _verify_ambulance_provider_primary_care_value
FROM _merge_six;

DROP VIEW IF EXISTS _merge_six;

In [0]:
CREATE OR REPLACE TABLE public_health.silver.facilities_all_merged_flagged AS
SELECT
  m.*,
  v._merge_source,
  CASE
    WHEN v._merge_source LIKE '%+%' THEN 'MATCHED'
    ELSE CONCAT('UNMATCHED: ', v._merge_source, ' only — no facility name match in other sources')
  END AS match_flag
FROM public_health.silver.facilities_hospital_infirmary_merged m
JOIN public_health.silver.verification v
  ON m.Facility_Name = v.Facility_Name;

SELECT
  Facility_Name,
  Health_Facility_Code,
  Region_Name,
  Province_Name,
  Ownership_Major_Classification,
  Health_Facility_Type,
  Service_Capability,
  Bed_Capacity,
  _merge_source,
  match_flag
FROM public_health.silver.facilities_all_merged_flagged
ORDER BY
  CASE WHEN match_flag LIKE 'UNMATCHED%' THEN 1 ELSE 0 END,
  _merge_source,
  Facility_Name
LIMIT 20;

In [0]:
SELECT *
FROM public_health.silver.facilities_hospital_infirmary_merged
WHERE Facility_Name IN (
  SELECT Facility_Name FROM public_health.silver.verification
  WHERE _merge_source LIKE '%+%'
)
ORDER BY Facility_Name
LIMIT 20

In [0]:
SELECT *
FROM public_health.silver.verification
WHERE _merge_source LIKE '%+%'
ORDER BY _merge_source, _verify_region_status, Facility_Name
LIMIT 20

In [0]:
CREATE OR REPLACE TABLE public_health.silver.facilities_hospital_unmatched AS
SELECT
  h.facility_name AS Facility_Name,
  h.nhfr_code AS NHFR_Code,
  h.region, h.province, h.city_municipality, h.address,
  h.ownership, h.ownership_class, h.hospital_class,
  h.service_capability, h.abc, h.add_on_services,
  h.blood_service_facility, h.rhiv_confirmatory_lab,
  h.ambulance_details, h.ambulance_type1_count, h.ambulance_type2_count, h.ambulance_total,
  h.ambulance_provider, h.dialysis_stations, h.dialysis_provider,
  h.ctf_classification, h.clinical_lab_class
FROM public_health.bronze.facilities_hfsrb_hospitals_bronze h
WHERE NOT EXISTS (
  SELECT 1 FROM public_health.bronze.facilities_nhfr_bronze n
  WHERE UPPER(TRIM(n.Facility_Name)) = UPPER(TRIM(h.facility_name))
)
AND NOT EXISTS (
  SELECT 1 FROM public_health.bronze.facilities_hfsrb_infirmaries_bronze i
  WHERE UPPER(TRIM(i.facility_name)) = UPPER(TRIM(h.facility_name))
)
AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_primary_care_bronze p WHERE UPPER(TRIM(p.facility_name)) = UPPER(TRIM(h.facility_name)))
AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_dialysis_clinics_bronze d WHERE UPPER(TRIM(d.facility_name)) = UPPER(TRIM(h.facility_name)))
AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_blood_service_bronze b WHERE UPPER(TRIM(b.facility_name)) = UPPER(TRIM(h.facility_name)));

In [0]:
SELECT
  Facility_Name, NHFR_Code, region, province, city_municipality,
  ownership, ownership_class, hospital_class, service_capability, abc
FROM public_health.silver.facilities_hospital_unmatched
ORDER BY region, province, Facility_Name
LIMIT 20

In [0]:
CREATE OR REPLACE TABLE public_health.silver.facilities_primary_care_unmatched AS
SELECT
  pc.facility_name AS Facility_Name,
  pc.nhfr_code AS NHFR_Code,
  pc.region, pc.province, pc.city_municipality,
  pc.ownership, pc.head_of_facility,
  pc.ambulance_arrangement, pc.ambulance_provider,
  pc.birthing_home_abc, pc.clinical_lab_capability,
  pc.pharmacy_arrangement, pc.xray_arrangement,
  pc.ancillary_services
FROM public_health.bronze.facilities_hfsrb_primary_care_bronze pc
WHERE NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_nhfr_bronze n WHERE UPPER(TRIM(n.Facility_Name)) = UPPER(TRIM(pc.facility_name)))
  AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_infirmaries_bronze i WHERE UPPER(TRIM(i.facility_name)) = UPPER(TRIM(pc.facility_name)))
  AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_hospitals_bronze h WHERE UPPER(TRIM(h.facility_name)) = UPPER(TRIM(pc.facility_name)))
  AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_dialysis_clinics_bronze d WHERE UPPER(TRIM(d.facility_name)) = UPPER(TRIM(pc.facility_name)))
  AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_blood_service_bronze b WHERE UPPER(TRIM(b.facility_name)) = UPPER(TRIM(pc.facility_name)));

SELECT Facility_Name, NHFR_Code, region, province, city_municipality, ownership
FROM public_health.silver.facilities_primary_care_unmatched
ORDER BY region, province, Facility_Name
LIMIT 20;

In [0]:
CREATE OR REPLACE TABLE public_health.silver.facilities_dialysis_unmatched AS
SELECT
  dc.facility_name AS Facility_Name,
  dc.nhfr_code AS NHFR_Code,
  dc.region, dc.province, dc.city_municipality,
  dc.ownership, dc.institutional_character,
  dc.stations, dc.head_of_facility
FROM public_health.bronze.facilities_hfsrb_dialysis_clinics_bronze dc
WHERE NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_nhfr_bronze n WHERE UPPER(TRIM(n.Facility_Name)) = UPPER(TRIM(dc.facility_name)))
  AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_infirmaries_bronze i WHERE UPPER(TRIM(i.facility_name)) = UPPER(TRIM(dc.facility_name)))
  AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_hospitals_bronze h WHERE UPPER(TRIM(h.facility_name)) = UPPER(TRIM(dc.facility_name)))
  AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_primary_care_bronze p WHERE UPPER(TRIM(p.facility_name)) = UPPER(TRIM(dc.facility_name)))
  AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_blood_service_bronze b WHERE UPPER(TRIM(b.facility_name)) = UPPER(TRIM(dc.facility_name)));

SELECT Facility_Name, NHFR_Code, region, province, city_municipality, ownership, stations
FROM public_health.silver.facilities_dialysis_unmatched
ORDER BY region, province, Facility_Name
LIMIT 20;

In [0]:
CREATE OR REPLACE TABLE public_health.silver.facilities_blood_service_unmatched AS
SELECT
  bs.facility_name AS Facility_Name,
  bs.nhfr_code AS NHFR_Code,
  bs.region, bs.province, bs.city_municipality,
  bs.ownership, bs.service_capability,
  bs.institutional_character, bs.head_of_laboratory
FROM public_health.bronze.facilities_hfsrb_blood_service_bronze bs
WHERE NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_nhfr_bronze n WHERE UPPER(TRIM(n.Facility_Name)) = UPPER(TRIM(bs.facility_name)))
  AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_infirmaries_bronze i WHERE UPPER(TRIM(i.facility_name)) = UPPER(TRIM(bs.facility_name)))
  AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_hospitals_bronze h WHERE UPPER(TRIM(h.facility_name)) = UPPER(TRIM(bs.facility_name)))
  AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_primary_care_bronze p WHERE UPPER(TRIM(p.facility_name)) = UPPER(TRIM(bs.facility_name)))
  AND NOT EXISTS (SELECT 1 FROM public_health.bronze.facilities_hfsrb_dialysis_clinics_bronze d WHERE UPPER(TRIM(d.facility_name)) = UPPER(TRIM(bs.facility_name)));

SELECT Facility_Name, NHFR_Code, region, province, city_municipality, ownership, service_capability
FROM public_health.silver.facilities_blood_service_unmatched
ORDER BY region, province, Facility_Name
LIMIT 20;